# Tự kiểm thử M1 trên Google Colab (CPU)

Notebook chạy Engine `src/crawl/` của `main` trên một mẫu nhỏ, trong môi trường giống testbed
của đề (Colab CPU-only). Mục tiêu là kiểm chứng **môi trường Colab và cách lưu kết quả**,
chưa phải benchmark toàn Range.

1. Clone repo tại `REF` (nhánh hoặc SHA), ghi commit thực tế vào `run.json`.
2. Cài `requirements.txt`, chạy `pytest` (log ra file, lỗi thì dừng).
3. Chọn ID: danh sách cụ thể, hoặc `N_IDS` ID từ `member_ids(MEMBER)` quanh ranh giới tháng.
4. Crawl cold qua harness `src.benchmark` vào `/content/` (đo RAM cả cây tiến trình, disk).
5. Resume ngay trong phiên **với mạng bị chặn**, kèm assertion.
6. `make_stats`, validator, số lần HTTP 429 và tổng thời gian chờ backoff.
7. Nén cả lượt chạy (dữ liệu, checkpoint, stats, log) thành **một** file `.tar.gz` kèm sha256,
   chép lên Drive.

**Phiên mới:** đặt `RESTORE_ARCHIVE` là đường dẫn file `.tar.gz` trên Drive rồi chạy lại từ
đầu. Notebook kiểm sha256, giải nén về đúng chỗ cũ trong `/content/`, rồi chạy resume và toàn
bộ assertion lần nữa.

**Bí mật:** mở 🔑 *Secrets* ở thanh bên trái Colab, thêm `CONTACT_EMAIL` (bắt buộc, arXiv
yêu cầu User-Agent có email) và `S2_API_KEY` (tùy chọn), bật *Notebook access*. Không dán
token hay key vào ô code.

**Lưu trữ:** crawl vào `/content/` để đo hiệu năng local. `/content/` mất khi runtime kết
thúc, nên không dùng làm nơi lưu lâu dài. Không crawl thẳng vào Drive: đọc/ghi nhiều file
nhỏ trên Drive chậm và có thể chạm giới hạn I/O
([Colab FAQ](https://research.google.com/colaboratory/faq.html)). Mỗi lượt có `RUN_ID`
riêng; thư mục đã tồn tại thì notebook dừng, không ghi đè. Không có nút xóa dữ liệu.

In [ ]:
# ==== Tham số ====
REPO_URL = "https://github.com/hcmus-arxiv-reference/arxiv-reference-matching.git"
REF = "main"            # nhánh hoặc SHA; dùng SHA để chạy lại đúng y một lượt cũ

ID_MODE = "explicit"    # "explicit": EXPLICIT_IDS; "member": N_IDS ID của MEMBER
EXPLICIT_IDS = ["2409.20569", "2410.00001", "2410.08025"]   # smoke test 03/11 trên máy Bảo
MEMBER = "minh"         # tên trong configs/ranges.toml
N_IDS = 20              # với Range nhiều Đoạn: lấy quanh ranh giới Đoạn đầu tiên

RUN_PYTEST = True
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/arxiv-m1-selftest"
RESTORE_ARCHIVE = None  # phiên mới: ví dụ f"{DRIVE_DIR}/<RUN_ID>__<SESSION>.tar.gz"

WORK_ROOT = "/content/m1-selftest"
REPO_DIR = "/content/arxiv-reference-matching"
BLOCKED_PROXY = "http://127.0.0.1:9"   # cổng discard: mọi request HTTP(S) đều thất bại
CRAWL_ARGS = ["--max-passes", "2", "--retry-delay", "60"]

In [ ]:
# Hàm dùng chung. Mọi lệnh ghi stdout/stderr ra file log để tải về khi lỗi.
import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, time
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SESSION = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
PENDING_LOGS = Path(WORK_ROOT, "sessions", SESSION)   # log trước khi có RUN_DIR
PENDING_LOGS.mkdir(parents=True, exist_ok=True)


def run(cmd, log, *, check=True, env=None, cwd=None):
    """Chạy lệnh, ghi log; check=True thì mã thoát khác 0 sẽ ném lỗi kèm 20 dòng cuối."""
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    with log.open("w", encoding="utf-8") as stream:
        stream.write("$ " + " ".join(map(str, cmd)) + "\n")
        stream.flush()
        result = subprocess.run(list(map(str, cmd)), stdout=stream, stderr=subprocess.STDOUT,
                                env=env, cwd=cwd, text=True)
    seconds = round(time.monotonic() - started, 2)
    print(f"exit={result.returncode} {seconds}s log={log}")
    if check and result.returncode != 0:
        tail = "\n".join(log.read_text(encoding="utf-8").splitlines()[-20:])
        raise RuntimeError(f"Lệnh thất bại (exit {result.returncode}): {cmd}\n{tail}")
    return {"cmd": list(map(str, cmd)), "returncode": result.returncode,
            "seconds": seconds, "log": str(log)}


def sha256_file(path):
    with open(path, "rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def data_checksum(data_dir):
    """sha256 trên danh sách file dữ liệu đã sắp xếp: raw/, metadata/, references/.

    Bỏ manifests/ và tables/: checkpoint và bảng xuất lại được ghi lại hợp lệ mỗi lượt.
    """
    data_dir = Path(data_dir)
    files = sorted(p for d in ("raw", "metadata", "references") for p in (data_dir / d).rglob("*")
                   if p.is_file())
    digest = hashlib.sha256()
    for path in files:
        digest.update(f"{path.relative_to(data_dir).as_posix()}\0{sha256_file(path)}\n".encode())
    return {"files": len(files), "sha256": digest.hexdigest()}


def task_state(data_dir):
    """Bản ghi cuối của mỗi task trong crawl.jsonl: (status, attempts, checksum)."""
    state = {}
    for line in Path(data_dir, "manifests", "crawl.jsonl").read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        state[f"{r['arxiv_id']}|{r['version']}|{r['kind']}"] = [r["status"], r["attempts"], r["checksum"]]
    return state


def http_retries(data_dir):
    """429/5xx và thời gian chờ từ manifests/http-events.jsonl (chỉ tải source arXiv)."""
    path = Path(data_dir, "manifests", "http-events.jsonl")
    events = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()] if path.exists() else []
    retries = [e for e in events if e["event"] == "retry"]
    return {"http_429": sum(e.get("status") == 429 for e in retries),
            "retries": len(retries),
            "backoff_seconds": round(sum(e.get("delay", 0) for e in retries), 1),
            "partial": sum(e["event"] == "partial" for e in events),
            "resume": sum(e["event"] == "resume" for e in events),
            "scope": "source downloads only; discovery/S2 retries are not logged here"}

In [ ]:
# Clone repo tại REF và ghi commit thực tế (SHA đầy đủ).
if not os.path.isdir(REPO_DIR):
    run(["git", "clone", REPO_URL, REPO_DIR], PENDING_LOGS / "git-clone.log")
run(["git", "-C", REPO_DIR, "fetch", "origin", REF], PENDING_LOGS / "git-fetch.log")
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"], PENDING_LOGS / "git-checkout.log")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("ref:", REF, "commit:", COMMIT)

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], PENDING_LOGS / "pip.log")

In [ ]:
# Bí mật: Colab Secrets (hoặc biến môi trường khi chạy ngoài Colab). Không in giá trị.
def secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

for name in ("CONTACT_EMAIL", "S2_API_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)
    print(f"{name}: {'đã đặt' if value else 'chưa đặt'}")
assert os.environ.get("CONTACT_EMAIL"), "Thêm CONTACT_EMAIL trong Colab Secrets trước khi crawl"
for name in ("HTTPS_PROXY", "HTTP_PROXY", "https_proxy", "http_proxy"):
    os.environ.pop(name, None)   # lượt cold cần mạng thật

In [ ]:
# Thư mục lượt chạy. Lượt mới: RUN_ID mới, đã tồn tại thì dừng. Phiên mới: khôi phục bản lưu.
import psutil

MACHINE = {"platform": platform.platform(), "python": sys.version, "cpus": os.cpu_count(),
           "ram_gib": round(psutil.virtual_memory().total / 2**30, 1),
           "disk_free_gib": round(shutil.disk_usage(WORK_ROOT).free / 2**30), "colab": IN_COLAB}
if SAVE_TO_DRIVE or RESTORE_ARCHIVE:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

RUNS = Path(WORK_ROOT, "runs")
if RESTORE_ARCHIVE:
    archive = Path(RESTORE_ARCHIVE)
    expected = Path(f"{archive}.sha256").read_text().split()[0]
    actual = sha256_file(archive)
    assert actual == expected, f"sha256 không khớp: {actual} != {expected}"
    with tarfile.open(archive) as tar:
        top = {Path(m.name).parts[0] for m in tar.getmembers()}
        assert len(top) == 1, f"Bản lưu phải có đúng một thư mục gốc, có {top}"
        RUN_ID = top.pop()
        RUN_DIR = RUNS / RUN_ID
        if RUN_DIR.exists():
            raise FileExistsError(f"{RUN_DIR} đã tồn tại; dùng runtime mới hoặc đổi WORK_ROOT")
        RUNS.mkdir(parents=True, exist_ok=True)
        tar.extractall(RUNS, filter="data")
    RUN = json.loads((RUN_DIR / "run.json").read_text(encoding="utf-8"))
    print("Khôi phục", RUN_ID, "từ", archive, "sha256 OK")
else:
    RUN_ID = f"{SESSION}_{COMMIT[:7]}"
    RUN_DIR = RUNS / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)   # không bao giờ ghi đè lượt cũ
    RUN = {"run_id": RUN_ID, "repo": REPO_URL, "sessions": []}
DATA_DIR = RUN_DIR / "data"
SESSION_DIR = RUN_DIR / "sessions" / SESSION
SESSION_DIR.mkdir(parents=True, exist_ok=False)
for log in PENDING_LOGS.iterdir():
    shutil.move(str(log), SESSION_DIR / log.name)
THIS = {"session": SESSION, "ref": REF, "commit": COMMIT, "machine": MACHINE,
        "restored_from": str(RESTORE_ARCHIVE) if RESTORE_ARCHIVE else None,
        "s2_api_key": bool(os.environ.get("S2_API_KEY")), "steps": {}}
RUN["sessions"].append(THIS)
if RESTORE_ARCHIVE and RUN["sessions"][0]["commit"] != COMMIT:
    print("CHÚ Ý: commit phiên này khác commit lượt cold:", RUN["sessions"][0]["commit"])


def save_run():
    (RUN_DIR / "run.json").write_text(json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")

save_run()
print("RUN_ID:", RUN_ID)
print(json.dumps(MACHINE, indent=2))

In [ ]:
if RUN_PYTEST:
    env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    THIS["steps"]["pytest"] = run([sys.executable, "-m", "pytest", "-q"], SESSION_DIR / "pytest.log",
                                  check=False, env=env)
    save_run()
    lines = Path(THIS["steps"]["pytest"]["log"]).read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[-30:] if THIS["steps"]["pytest"]["returncode"] else lines[-1:]))
    assert THIS["steps"]["pytest"]["returncode"] == 0, "pytest thất bại; xem log (kiểm phiên bản Python trước)"

In [ ]:
# Chọn ID. Lượt khôi phục dùng lại đúng danh sách đã lưu.
from src.common.config import load_ranges


def around_boundary(member, n):
    """n ID liên tiếp quanh ranh giới Đoạn đầu tiên; Range một Đoạn thì lấy n ID đầu."""
    segments = load_ranges()[member].range.segments
    ids = [i for s in segments for i in s.ids()]
    if len(segments) == 1:
        return ids[:n]
    boundary = len(segments[0])
    start = max(0, boundary - n // 2)
    picked = ids[start:start + n]
    assert picked[0] in segments[0] and picked[-1] in segments[1], "chưa phủ hai bên ranh giới"
    return picked


if RESTORE_ARCHIVE:
    IDS = RUN["ids"]
else:
    if ID_MODE == "explicit":
        IDS, mode = list(dict.fromkeys(EXPLICIT_IDS)), "explicit"
    elif ID_MODE == "member":
        IDS, mode = around_boundary(MEMBER, N_IDS), f"member:{MEMBER}:{N_IDS}"
    else:
        raise ValueError(f"ID_MODE không hợp lệ: {ID_MODE}")
    RUN.update(id_mode=mode, ids=IDS)
IDS_FILE = RUN_DIR / "ids.txt"
if not IDS_FILE.exists():
    IDS_FILE.write_text("\n".join(IDS) + "\n", encoding="utf-8")
save_run()
print(RUN["id_mode"], len(IDS), "ID:", IDS[0], "…", IDS[-1])

In [ ]:
# Crawl chạy qua harness đo RAM/disk. check=False: rẽ nhánh theo cả mã thoát lẫn status.
def crawl(label, env):
    out = SESSION_DIR / f"bench-{label}"
    cmd = [sys.executable, "-m", "src.benchmark", "--data-dir", DATA_DIR, "--out", out, "--",
           sys.executable, "-m", "src.milestone1_crawler", "--ids-file", IDS_FILE,
           "--data-dir", DATA_DIR, *CRAWL_ARGS]
    step = run(cmd, SESSION_DIR / f"crawl-{label}.log", check=False, env=env)
    report = json.loads((DATA_DIR / "manifests" / "range-run.json").read_text(encoding="utf-8"))
    step.update(status=report["status"], visited_this_pass=report["visited_this_pass"],
                counts=report["counts"], benchmark=json.loads((out / "benchmark.json").read_text()))
    return step


if not RESTORE_ARCHIVE:
    cold = crawl("cold", dict(os.environ))
    cold["http"] = http_retries(DATA_DIR)
    THIS["steps"]["cold"] = cold
    RUN["cold_complete"] = cold["returncode"] == 0 and cold["status"] == "complete"
    save_run()
    print({k: cold[k] for k in ("returncode", "status", "seconds", "counts", "http")})
    if not RUN["cold_complete"]:
        print("Lượt cold CHƯA hoàn tất: bỏ qua assertion resume, xem chẩn đoán ở các ô dưới.")

In [ ]:
# Resume với mạng bị chặn (proxy tới cổng discard). Đạt hết assertion => lượt này không cần mạng.
def resume_check(label):
    before = {"data": data_checksum(DATA_DIR), "tasks": task_state(DATA_DIR)}
    env = dict(os.environ, HTTPS_PROXY=BLOCKED_PROXY, HTTP_PROXY=BLOCKED_PROXY,
               https_proxy=BLOCKED_PROXY, http_proxy=BLOCKED_PROXY)
    env["NO_PROXY"] = env["no_proxy"] = ""  # không cho host bỏ qua proxy chặn mạng
    step = crawl(label, env)
    after = {"data": data_checksum(DATA_DIR), "tasks": task_state(DATA_DIR)}
    changed = sorted(k for k in before["tasks"].keys() | after["tasks"].keys()
                     if before["tasks"].get(k) != after["tasks"].get(k))
    step["checks"] = checks = {
        "exit_0": step["returncode"] == 0,
        "status_complete": step["status"] == "complete",
        "visited_0": step["visited_this_pass"] == 0,
        "tasks_unchanged": not changed,
        "data_checksum_unchanged": before["data"] == after["data"],
    }
    step["data_checksum"], step["changed_tasks"] = after["data"], changed[:20]
    THIS["steps"][label] = step
    save_run()
    print(json.dumps(checks, indent=2))
    assert all(checks.values()), f"Resume không đạt: {checks}; task đổi: {changed[:20]}"


if RUN.get("cold_complete"):
    resume_check("resume-restored" if RESTORE_ARCHIVE else "resume")
else:
    THIS["steps"]["resume"] = {"skipped": "cold run incomplete"}
    save_run()
    print("Bỏ qua resume vì lượt cold chưa hoàn tất.")

## Thống kê, validator và chẩn đoán

In [ ]:
from src.crawl.validation import validate_crawl


def segments_for(ids):
    """Gộp ID thành các Đoạn liên tiếp, dùng cho range.segments của validator."""
    segments = []
    for prefix, number in sorted((i.split(".")[0], int(i.split(".")[1])) for i in ids):
        if segments and segments[-1]["prefix"] == prefix and segments[-1]["end"] == number - 1:
            segments[-1]["end"] = number
        else:
            segments.append({"prefix": prefix, "start": number, "end": number})
    return segments


stats_dir = SESSION_DIR / "make_stats"
THIS["steps"]["make_stats"] = run([sys.executable, "-m", "src.make_stats", "--data-dir", DATA_DIR,
                                   "--ids-file", IDS_FILE, "--out", stats_dir],
                                  SESSION_DIR / "make_stats.log")
summary = json.loads((stats_dir / "summary.json").read_text())
cold_session = RUN["sessions"][0]
cold = cold_session["steps"].get("cold", {})
bench = cold.get("benchmark", {})
claim = {
    "schema_version": 1, "member": RUN.get("id_mode"), "commit": cold_session["commit"],
    "range": {"segments": segments_for(IDS)}, "ids_attempted": len(IDS),
    "run": {"cold_run": True, "machine": cold_session["machine"]["platform"], "python": cold_session["machine"]["python"],
            "s2_api_key": cold_session["s2_api_key"],
            "measured_by": "src.benchmark (psutil; RSS cả cây tiến trình, disk qua os.walk)"},
    "totals": {"wall_seconds": bench.get("wall_seconds"), "peak_rss_mb": bench.get("peak_rss_mb"),
               "avg_rss_mb": bench.get("avg_rss_mb"), "peak_disk_mb": bench.get("peak_disk_mb")},
    "http": cold.get("http"),
    "validation_session": {"session": SESSION, "commit": COMMIT, "machine": MACHINE},
    "notes": f"Colab self-test {RUN_ID}; crawl vào /content/, không phải Drive.",
}
claim_path = SESSION_DIR / "benchmark.claim.json"
claim_path.write_text(json.dumps(claim, ensure_ascii=False, indent=2))
report = validate_crawl(DATA_DIR, claim_path)
THIS["steps"]["validator"] = {"ok": report["ok"], "findings": report["findings"]}
save_run()
print("validator ok:", report["ok"])
for item in report["findings"]:
    print("-", item["code"], item["message"])
{k: v for k, v in summary.items() if v is not None}

In [ ]:
# Publication chưa xong (nếu có): trạng thái từng task để chẩn đoán.
import csv
rows = list(csv.DictReader(open(stats_dir / "papers.csv", encoding="utf-8")))
print({s: sum(r["status"] == s for r in rows) for s in sorted({r["status"] for r in rows})})
pending = [r["id"] for r in rows if r["status"] != "done"]
state = task_state(DATA_DIR)
{i: {k.split("|", 1)[1]: v[:2] for k, v in state.items() if k.startswith(i + "|")} for i in pending[:20]}

In [ ]:
# Lưu: nén cả lượt chạy thành một file + sha256, chép lên Drive (tránh nhiều file nhỏ).
save_run()
ARCHIVES = Path(WORK_ROOT, "archives")
ARCHIVES.mkdir(parents=True, exist_ok=True)
archive = ARCHIVES / f"{RUN_ID}__{SESSION}.tar.gz"
with tarfile.open(archive, "x:gz") as tar:
    tar.add(RUN_DIR, arcname=RUN_ID)
digest = sha256_file(archive)
Path(f"{archive}.sha256").write_text(f"{digest}  {archive.name}\n")
print(archive, f"{archive.stat().st_size / 2**20:.1f} MiB", digest)
if SAVE_TO_DRIVE:
    for path in (archive, Path(f"{archive}.sha256")):
        target = Path(DRIVE_DIR, path.name)
        if target.exists():
            raise FileExistsError(target)
        shutil.copy2(path, target)
    assert sha256_file(Path(DRIVE_DIR, archive.name)) == digest
    print("Đã chép lên Drive:", Path(DRIVE_DIR, archive.name))
    print('Phiên mới: RESTORE_ARCHIVE = "' + str(Path(DRIVE_DIR, archive.name)) + '"')

In [ ]:
# Tải run.json về máy để dán vào báo cáo (dữ liệu crawl nằm trong bản lưu trên Drive).
if IN_COLAB:
    from google.colab import files
    files.download(str(RUN_DIR / "run.json"))